# Demo D3. The forced wave equation

**Partial differential equations · driven string.**

A string fixed at both ends and pushed by a steady distributed force obeys $u_{tt} = c^2 u_{xx} + F(x)$ on $x \in [0,1]$ with $u(0,t)=u(1,t)=0$. Here $F(x) = \sin(2\pi k x)$. Starting from rest ($u(x,0)=0$, $u_t(x,0)=0$), the string swings up toward a **steady shape** $v(x)$ and then oscillates about it. This lab integrates the equation with an explicit finite-difference scheme and compares the motion against that steady state.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, IntSlider, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Steady state and discretization

If the string settles, $u_{tt}=0$ and the steady shape solves $c^2 v'' = -\sin(2\pi k x)$ with $v(0)=v(1)=0$. Because $k$ is an integer this gives
$$ v(x) = \frac{\sin(2\pi k x)}{c^2 (2\pi k)^2}. $$
The force $\sin(2\pi k x)$ is exactly the fixed-end mode with $2k$ half-waves, so it feeds that mode directly; its steady amplitude falls off like $1/k^2$.

On a grid $x_i = i\,\Delta x$ the explicit leapfrog update is
$$ u_i^{n+1} = 2u_i^n - u_i^{n-1} + r^2\,(u_{i+1}^n - 2u_i^n + u_{i-1}^n) + \Delta t^2\,F(x_i), \qquad r = \frac{c\,\Delta t}{\Delta x}, $$
which is stable for $r \le 1$ (the CFL condition).

In [ ]:
# ---------------------------------------------------------------------------
# Grid and the explicit finite-difference integrator. u starts at rest, so the
# first step uses the u_t(x,0)=0 Taylor start; later steps use leapfrog.
# ---------------------------------------------------------------------------
Nx = 200
xg = np.linspace(0.0, 1.0, Nx + 1)
dxg = xg[1] - xg[0]
dtg = 0.002                                  # r = c*dt/dx = 0.4c <= 1 for c <= 2

def steady_state(c, k):
    return np.sin(2 * np.pi * k * xg) / (c**2 * (2 * np.pi * k) ** 2)

def evolve_forced(c, k, T):
    """March u_tt = c^2 u_xx + sin(2 pi k x) from rest to time T."""
    F = np.sin(2 * np.pi * k * xg)
    r2 = (c * dtg / dxg) ** 2
    nsteps = int(round(T / dtg))

    u_prev = np.zeros(Nx + 1)                # u(x, 0) = 0
    if nsteps == 0:
        return u_prev, F
    u_curr = np.zeros(Nx + 1)                # first step with u_t(x,0)=0
    u_curr[1:-1] = (0.5 * r2 * (u_prev[2:] - 2 * u_prev[1:-1] + u_prev[:-2])
                    + 0.5 * dtg**2 * F[1:-1])
    for _ in range(1, nsteps):
        u_next = np.zeros(Nx + 1)
        u_next[1:-1] = (2 * u_curr[1:-1] - u_prev[1:-1]
                        + r2 * (u_curr[2:] - 2 * u_curr[1:-1] + u_curr[:-2])
                        + dtg**2 * F[1:-1])
        u_prev, u_curr = u_curr, u_next
    return u_curr, F

## Watch it swing

Drag the **time** slider: from a flat start the string rises toward the steady shape (red dashed) and overshoots, oscillating about it. The grey curve is the driving force $F(x)$ (scaled to fit).

In [ ]:
def show_forced(c=1.0, k=1, t=1.0):
    u, F = evolve_forced(c, k, t)
    v = steady_state(c, k)

    amp = max(np.abs(v).max() * 2.5, np.abs(u).max(), 1e-6)
    plt.figure()
    plt.plot(xg, F * 0.5 * amp, color="0.8", lw=1.5, label=r"force $F(x)$ (scaled)")
    plt.plot(xg, v, "r--", lw=1.5, label="steady state $v(x)$")
    plt.plot(xg, u, "b-", lw=2.5, label=rf"$u(x,\,t={t:.2f})$")
    plt.axhline(0.0, color="0.7", lw=1)
    plt.scatter([0, 1], [0, 0], color="k", zorder=5)
    plt.xlim(0, 1); plt.ylim(-amp, amp)
    plt.xlabel("x"); plt.ylabel("displacement u")
    plt.title("Forced wave equation: motion about the steady state")
    plt.legend(loc="upper right"); plt.show()

interact(
    show_forced,
    c=FloatSlider(value=1.0, min=0.3, max=2.0, step=0.05, description="speed c"),
    k=IntSlider(value=1, min=1, max=5, step=1, description="mode k"),
    t=FloatSlider(value=1.0, min=0.0, max=4.0, step=0.05, description="time t"),
);

## Things to try

- At $t=0$ the string is flat; as $t$ grows it climbs toward $v(x)$ and overshoots, since nothing damps the motion.
- Raising $k$ adds half-waves to the force and shrinks the steady amplitude like $1/k^2$.
- Lowering the speed $c$ both slows the oscillation and enlarges the steady shape ($v \propto 1/c^2$).

## Summary

- A steadily forced, fixed-end string oscillates forever about the steady shape solving $c^2 v'' = -F$.
- The spatial force $\sin(2\pi k x)$ drives the $2k$ mode directly, with steady amplitude $\propto 1/(c^2 k^2)$.
- The explicit leapfrog scheme is stable under the CFL limit $r = c\,\Delta t/\Delta x \le 1$.